# Qwen-VL Fine-Tuning (Lighting Defects)

This notebook fine-tunes **Qwen-VL** with **QA-style labels** to describe lighting defects in renders.

**Assumptions**
- You are using a `uv` virtual environment.
- Your dataset is a JSONL file with keys: `image`, `question`, `answer`.


## 1) Install dependencies (uv)
Run this once in a terminal (not inside the notebook):
```bash
uv venv
uv pip install -r requirements.txt
```

## 2) Dataset format (JSONL)
Each line in your `train.jsonl` should look like:
```json
{"image":"/abs/path/to/image.jpg","question":"Describe lighting defects in this render.","answer":"Light is inverted; lamp illuminates floor instead of ceiling. Visible banding on left wall."}
```

In [ ]:
# 3) Fine-tuning script (LoRA)
import os
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# Config
MODEL_ID = "Qwen/Qwen-VL-Chat"
DATA_PATH = "./data/train.jsonl"
OUTPUT_DIR = "./qwen_vl_lighting_defects"

# Load dataset
train_ds = load_dataset("json", data_files=DATA_PATH, split="train")

# Load processor & model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Add LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, lora_config)

MAX_LEN = 512

def preprocess(batch):
    images = [img for img in batch["image"]]
    questions = batch["question"]
    answers = batch["answer"]

    prompts = [f"User: <image>\n{q}\nAssistant:" for q in questions]

    inputs = processor(
        text=prompts,
        images=images,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    with processor.as_target_processor():
        labels = processor(
            text=answers,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

    inputs["labels"] = labels["input_ids"]
    return inputs

train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)


## 4) Inference example

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch

MODEL_DIR = "./qwen_vl_lighting_defects"
processor = AutoProcessor.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)

image_path = "./sample.jpg"
question = "Describe lighting defects in this render."

prompt = f"User: <image>\n{question}\nAssistant:"
inputs = processor(text=prompt, images=image_path, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=128)
print(processor.decode(output[0], skip_special_tokens=True))
